# Internal Wiki Assistant

**Project 19** — Core RAG Projects

Build a retrieval assistant that ingests markdown/wiki-like documents, creates
a clean searchable index with document metadata, and answers questions using
the most relevant wiki sections.

**Key Skills:** Markdown parsing, heading-aware chunking, document metadata
extraction, section-level indexing, breadcrumb navigation, metadata-boosted
retrieval.

## Project Overview

Every organization has a wiki — Confluence, Notion, GitBook, or plain markdown
files in a repo. These wikis grow organically and become hard to search:

- **Keyword search fails** — users search "deploy to prod" but the wiki says
  "production release process"
- **Flat search ignores structure** — wiki pages have headings, sub-sections,
  and hierarchy. A search hit in an H1 title is more relevant than one buried
  in a sub-sub-section
- **No metadata awareness** — the "Onboarding" page last updated 3 years ago
  is probably stale

This notebook builds an **Internal Wiki Assistant** that:

1. **Parses markdown documents** preserving heading structure
2. **Chunks by section** respecting heading hierarchy (never splits mid-section)
3. **Extracts metadata** — author, last updated, department, tags
4. **Builds a clean index** with section-level granularity
5. **Retrieves with metadata boosting** — recent docs rank higher, tagged docs
   get boosted for matching queries
6. **Returns answers with breadcrumb paths** — "Engineering > Deployments > Rollback Procedure"

### Why Section-Level Indexing Matters

Most wiki pages are long (1000-5000 words). Indexing the full page means:
- Embedding quality degrades (too much text for one vector)
- Results point to a whole page, not the specific answer
- Irrelevant sections on the same page dilute the signal

**Section-level indexing** splits at heading boundaries, so each chunk is
a focused topic with its heading path as context.

## Learning Goals

By the end of this notebook you will understand:

- How to parse markdown into a structured document tree
- How to chunk documents by heading hierarchy without breaking sections
- How to extract and use document metadata for retrieval boosting
- How breadcrumb paths improve result context and navigation
- How to combine semantic similarity with metadata signals (recency, tags)
- When section-level indexing beats page-level indexing

## Problem Statement

**Scenario:** A mid-size tech company has 20+ wiki pages covering engineering
processes, HR policies, product specs, and team norms. Employees waste time
searching for answers that exist in the wiki but are hard to find.

**Goal:** Build a wiki assistant that:
1. Ingests markdown documents with metadata (author, date, department, tags)
2. Parses heading structure to create section-level chunks
3. Finds the most relevant section for any natural-language question
4. Returns the answer with its full heading path (breadcrumb)
5. Uses recency and tag metadata to improve ranking

## Why This Project Matters

| Pain Point | Impact |
|------------|--------|
| "I know it's in the wiki somewhere" | Average 15 min searching per query |
| Stale documentation | Teams follow outdated processes |
| No section-level search | Results link to 3000-word pages |
| New employee onboarding | Takes weeks to learn where info lives |
| Cross-team discovery | Teams don't know other teams' documented processes |

A well-indexed wiki assistant **reduces search time from 15 minutes to
15 seconds** and surfaces the exact section — not just the page.

## Environment Setup

In [1]:
import os
import re
import json
import hashlib
import textwrap
import warnings
from collections import Counter, defaultdict
from datetime import datetime, timedelta

import numpy as np
warnings.filterwarnings("ignore")

# ── Optional dependencies ──
USE_CHROMA = False
USE_ST = False

try:
    import chromadb
    USE_CHROMA = True
    print("[OK] chromadb available")
except ImportError:
    print("[INFO] chromadb not available - using TF-IDF fallback")

try:
    from sentence_transformers import SentenceTransformer
    USE_ST = True
    print("[OK] sentence-transformers available")
except ImportError:
    print("[INFO] sentence-transformers not available - using TF-IDF fallback")

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("\nSetup complete.")

[OK] chromadb available


[OK] sentence-transformers available

Setup complete.


In [2]:
# ── Configuration ──
TOP_K = 5
SIMILARITY_THRESHOLD = 0.2
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))

# Metadata boosting weights
RECENCY_BOOST = 0.05      # bonus per year of recency (vs 5yr baseline)
TAG_MATCH_BOOST = 0.10    # bonus when query matches a document tag
HEADING_DEPTH_PENALTY = 0.02  # slight penalty per heading depth level

print(f"Top-K: {TOP_K}, Similarity threshold: {SIMILARITY_THRESHOLD}")
print(f"Recency boost: {RECENCY_BOOST}/yr, Tag match boost: {TAG_MATCH_BOOST}")

Top-K: 5, Similarity threshold: 0.2
Recency boost: 0.05/yr, Tag match boost: 0.1


## Internal Wiki Document Corpus

We create **8 realistic wiki pages** covering different departments. Each page
is formatted as markdown with:
- A YAML-like metadata header (author, last_updated, department, tags)
- Hierarchical headings (H1, H2, H3)
- Realistic content with code blocks, lists, and tables where appropriate

These simulate a real internal wiki for a tech company called **Acme Corp**.

In [3]:
# ── Wiki pages as markdown documents ──

WIKI_PAGES = [
    {
        "page_id": "wiki-001",
        "filename": "deployment-guide.md",
        "metadata": {
            "title": "Production Deployment Guide",
            "author": "Sarah Chen",
            "last_updated": "2026-03-15",
            "department": "Engineering",
            "tags": ["deployment", "production", "CI/CD", "rollback"],
        },
        "content": """# Production Deployment Guide

## Overview

This guide covers Acme Corp's production deployment process. All deployments
follow our CI/CD pipeline and require approval from a team lead.

## Pre-Deployment Checklist

Before initiating a deployment:

1. All tests pass on the main branch (unit, integration, e2e)
2. Code review approved by at least 2 reviewers
3. Changelog updated with version number
4. Feature flags configured for gradual rollout
5. On-call engineer confirmed and available
6. Rollback plan documented

## Deployment Steps

### Step 1: Create Release Branch

Create a release branch from main:
```
git checkout -b release/v2.5.0 main
```

Tag the release and push:
```
git tag v2.5.0
git push origin v2.5.0
```

### Step 2: Deploy to Staging

The CI/CD pipeline automatically deploys tagged releases to staging.
Verify on staging for at least 30 minutes before promoting.

### Step 3: Production Promotion

Use the deployment dashboard to promote from staging to production.
Select canary deployment (10% traffic for 1 hour, then full rollout).

## Rollback Procedure

If issues are detected after deployment:

1. Go to the deployment dashboard
2. Click "Rollback" on the current deployment
3. Select the previous stable version
4. Confirm rollback (takes ~5 minutes)
5. Notify the team in #engineering-alerts

Rollbacks are instant for configuration changes but take 5-10 minutes
for code deployments due to container restart time.

## Deployment Windows

- **Standard deployments:** Tuesday-Thursday, 10am-4pm PT
- **Hotfixes:** Any time, but require VP Engineering approval outside business hours
- **Database migrations:** Wednesday 2-4pm PT only (DBA must be present)
- **No deployments:** Friday after 2pm, weekends, company holidays

## Emergency Deployment Process

For critical production issues (P0/P1):

1. Get verbal approval from VP Engineering or CTO
2. Create hotfix branch from the last stable release tag
3. Minimum 1 code review (can be done in real-time)
4. Deploy directly to production (skip staging soak time)
5. File incident report within 24 hours
""",
    },
    {
        "page_id": "wiki-002",
        "filename": "onboarding-guide.md",
        "metadata": {
            "title": "New Employee Onboarding Guide",
            "author": "Mike Johnson",
            "last_updated": "2026-01-10",
            "department": "HR",
            "tags": ["onboarding", "new-hire", "setup", "orientation"],
        },
        "content": """# New Employee Onboarding Guide

## Welcome to Acme Corp!

Congratulations on joining the team. This guide will help you get set up
and productive in your first two weeks.

## Week 1: Getting Started

### Day 1: Orientation

- 9:00am — Welcome meeting with your manager
- 10:00am — HR orientation (benefits, policies, safety)
- 11:00am — IT setup (laptop, accounts, access badges)
- 1:00pm — Team lunch
- 2:00pm — Workspace setup and tool installation
- 3:00pm — Meet your onboarding buddy

### Day 2-3: Environment Setup

Follow the Developer Environment Setup guide to install:
- IDE (VS Code recommended, JetBrains also supported)
- Docker Desktop
- Node.js 20 LTS and Python 3.12+
- Git and configure SSH keys
- VPN client (GlobalProtect)
- Slack, Zoom, and Google Workspace

### Day 4-5: Learning the Codebase

- Clone the main repositories (see Repository Map)
- Complete the "First PR" tutorial
- Shadow a team member on their current task
- Attend team standup and sprint planning

## Week 2: Getting Productive

### Meetings to Schedule

- 1:1 with your manager (recurring weekly)
- Meet-and-greet with each team member (30 min each)
- Architecture overview session with tech lead
- Product demo with product manager

### First Tasks

Your manager will assign 2-3 starter tasks from the "good-first-issue"
label in our issue tracker. These are designed to be small, well-scoped,
and educational about our codebase.

## Access and Permissions

### Systems You'll Need Access To

| System | How to Request | Approval |
|--------|---------------|----------|
| GitHub org | IT ticket | Automatic |
| AWS Console | IT ticket + manager approval | 1 business day |
| Production DB (read) | IT ticket + tech lead approval | 2 business days |
| Production DB (write) | IT ticket + VP Eng approval | 3 business days |
| PagerDuty | Engineering ops setup | After on-call training |

### VPN Access

VPN is required for accessing internal services from outside the office.
Install GlobalProtect and use your Okta credentials to connect.
If you have VPN issues, contact IT Support at it-help@acmecorp.com.

## Benefits Overview

### Health Insurance

- Medical: Blue Shield PPO (company pays 90% of premium)
- Dental: Delta Dental (company pays 100%)
- Vision: VSP (company pays 100%)
- Open enrollment is in November each year

### Time Off

- 20 days PTO per year (accrued monthly)
- 10 company holidays
- 5 sick days (separate from PTO)
- Parental leave: 16 weeks paid

### Other Benefits

- 401(k) with 4% company match (vested immediately)
- $1,500/year learning and development budget
- $500/year home office stipend
- Free lunch on Tuesdays and Thursdays
""",
    },
    {
        "page_id": "wiki-003",
        "filename": "api-design-standards.md",
        "metadata": {
            "title": "API Design Standards",
            "author": "Priya Patel",
            "last_updated": "2025-11-20",
            "department": "Engineering",
            "tags": ["API", "REST", "standards", "design", "backend"],
        },
        "content": """# API Design Standards

## Purpose

This document defines Acme Corp's standards for designing REST APIs. All new
API endpoints must follow these conventions for consistency and maintainability.

## URL Conventions

### Resource Naming

- Use plural nouns for collections: `/users`, `/orders`, `/products`
- Use kebab-case for multi-word resources: `/order-items`, `/user-profiles`
- Nest related resources: `/users/{id}/orders`
- Maximum nesting depth: 2 levels

### Versioning

All APIs must be versioned using URL path versioning:
```
/api/v1/users
/api/v2/users
```

### Query Parameters

- Pagination: `?page=1&per_page=20` (default per_page=20, max=100)
- Sorting: `?sort=created_at&order=desc`
- Filtering: `?status=active&role=admin`
- Search: `?q=search+term`

## Request and Response Format

### HTTP Methods

| Method | Usage | Idempotent |
|--------|-------|:---:|
| GET | Retrieve resource(s) | Yes |
| POST | Create new resource | No |
| PUT | Full resource update | Yes |
| PATCH | Partial update | No |
| DELETE | Remove resource | Yes |

### Response Codes

- 200: Success (GET, PUT, PATCH)
- 201: Created (POST)
- 204: No Content (DELETE)
- 400: Bad Request (validation error)
- 401: Unauthorized (missing/invalid auth)
- 403: Forbidden (insufficient permissions)
- 404: Not Found
- 409: Conflict (duplicate resource)
- 422: Unprocessable Entity (semantic error)
- 429: Too Many Requests (rate limited)
- 500: Internal Server Error

### Error Response Format

All error responses must use this format:
```json
{
  "error": {
    "code": "VALIDATION_ERROR",
    "message": "Human-readable error description",
    "details": [
      {"field": "email", "message": "Invalid email format"}
    ]
  }
}
```

## Authentication

All APIs use Bearer token authentication via Okta:
```
Authorization: Bearer <access_token>
```

Internal service-to-service calls use mTLS with client certificates.

## Rate Limiting

Default rate limits:
- Public APIs: 100 requests/minute per API key
- Internal APIs: 1000 requests/minute per service
- Admin APIs: 50 requests/minute per user

Rate limit headers are included in every response:
```
X-RateLimit-Limit: 100
X-RateLimit-Remaining: 95
X-RateLimit-Reset: 1620000000
```

## Pagination

All list endpoints must support cursor-based pagination:
```json
{
  "data": [...],
  "pagination": {
    "next_cursor": "abc123",
    "has_more": true
  }
}
```

Offset-based pagination (`?page=1`) is acceptable for internal tools
but not for public APIs due to performance concerns with large datasets.
""",
    },
    {
        "page_id": "wiki-004",
        "filename": "incident-response.md",
        "metadata": {
            "title": "Incident Response Playbook",
            "author": "David Kim",
            "last_updated": "2026-02-28",
            "department": "Engineering",
            "tags": ["incident", "on-call", "postmortem", "SLA", "alerting"],
        },
        "content": """# Incident Response Playbook

## Severity Levels

### P0 — Critical

- Complete service outage or data loss
- Revenue impact > $10K/hour
- Response time: 15 minutes
- Resolution target: 1 hour
- Communication: Every 30 minutes to stakeholders

### P1 — High

- Major feature degraded, no workaround
- Revenue impact > $1K/hour
- Response time: 30 minutes
- Resolution target: 4 hours
- Communication: Every 1 hour to stakeholders

### P2 — Medium

- Feature degraded with workaround available
- Limited user impact
- Response time: 2 hours
- Resolution target: 1 business day
- Communication: Status page update

### P3 — Low

- Minor issue, cosmetic, non-blocking
- Response time: Next business day
- Resolution target: 1 week

## On-Call Responsibilities

### Primary On-Call

- Monitor alerts in PagerDuty
- Acknowledge pages within 5 minutes
- Triage and assess severity
- Engage secondary on-call if needed
- Update status page for P0/P1

### Secondary On-Call

- Available as backup for primary
- Help with investigation when engaged
- Take over if primary is unavailable

### On-Call Rotation

- Rotation: Weekly, Monday 9am to Monday 9am
- Schedule: Published 4 weeks in advance
- Swaps: Arrange via #oncall-swaps Slack channel
- Compensation: $500/week on-call stipend + $200 per P0/P1 incident

## Incident Communication

### Internal Communication

1. Post in #incidents Slack channel with severity and description
2. Create incident channel: #inc-YYYY-MM-DD-short-description
3. Assign Incident Commander (IC) role
4. Post updates every 30 min (P0) or 60 min (P1)

### External Communication

For P0/P1 incidents affecting customers:

1. Update status page (status.acmecorp.com) within 15 minutes
2. Post initial customer communication within 30 minutes
3. Send email update to affected customers for P0
4. Post resolution update when fixed

## Postmortem Process

Required for all P0 and P1 incidents.

### Timeline

- Draft postmortem: Within 3 business days
- Review meeting: Within 5 business days
- Action items assigned: During review meeting
- Action items completed: Within 2 sprints

### Postmortem Template

1. Incident summary (1-2 sentences)
2. Timeline of events
3. Root cause analysis (use 5-Whys)
4. Impact assessment (users affected, duration, revenue)
5. What went well
6. What could be improved
7. Action items with owners and deadlines

### Blameless Culture

Postmortems are blameless. Focus on systems, processes, and tooling —
never on individuals. The goal is to prevent recurrence, not assign blame.
""",
    },
    {
        "page_id": "wiki-005",
        "filename": "data-privacy-policy.md",
        "metadata": {
            "title": "Data Privacy and Security Policy",
            "author": "Lisa Wang",
            "last_updated": "2026-04-01",
            "department": "Legal",
            "tags": ["privacy", "security", "GDPR", "PII", "compliance"],
        },
        "content": """# Data Privacy and Security Policy

## Scope

This policy applies to all Acme Corp employees, contractors, and third-party
vendors who handle customer data or internal sensitive information.

## Data Classification

### Public

- Marketing materials, blog posts, public documentation
- No restrictions on sharing

### Internal

- Company-wide announcements, internal processes, non-sensitive metrics
- Share within the company only, not externally

### Confidential

- Customer data, financial records, employee personal information
- Need-to-know basis only, must be encrypted at rest and in transit

### Restricted

- Credentials, encryption keys, security audit reports
- Access requires VP+ approval, logged access, no copies

## Personal Data Handling (GDPR/CCPA)

### What Counts as Personal Data

- Name, email, phone number
- IP addresses and device identifiers
- Location data
- Payment information
- Any data that can identify a specific person

### Data Processing Rules

1. Collect only what is necessary (data minimization)
2. Explain data use in plain language (transparency)
3. Get explicit consent before collecting (consent)
4. Allow users to export their data (portability)
5. Allow users to request deletion (right to erasure)
6. Notify within 72 hours of a breach (breach notification)

### Data Retention

| Data Type | Retention Period | After Retention |
|-----------|-----------------|-----------------|
| Account data | Duration of account + 30 days | Deleted |
| Usage logs | 12 months | Anonymized |
| Payment records | 7 years (legal requirement) | Archived securely |
| Support tickets | 3 years | Deleted |
| Marketing consent | Until withdrawn | Deleted immediately |

## Security Requirements

### Authentication

- Multi-factor authentication (MFA) required for all internal systems
- Passwords: minimum 12 characters, rotated every 90 days
- SSO via Okta for all supported applications
- Service accounts require key rotation every 60 days

### Data Encryption

- At rest: AES-256 for all databases and storage
- In transit: TLS 1.3 minimum
- PII fields: Application-level encryption with separate key management
- Backups: Encrypted with separate key from production

### Access Controls

- Principle of least privilege
- Quarterly access reviews for all systems
- Immediate revocation on employee departure
- Audit log for all access to confidential/restricted data

## Incident Reporting

If you suspect a data breach or security incident:

1. Do NOT attempt to investigate on your own
2. Report immediately to security@acmecorp.com
3. Do NOT discuss the incident outside the response team
4. Preserve all evidence (do not delete logs or emails)
5. Security team will activate incident response within 1 hour
""",
    },
    {
        "page_id": "wiki-006",
        "filename": "architecture-overview.md",
        "metadata": {
            "title": "System Architecture Overview",
            "author": "James Taylor",
            "last_updated": "2025-09-15",
            "department": "Engineering",
            "tags": ["architecture", "microservices", "infrastructure", "AWS"],
        },
        "content": """# System Architecture Overview

## High-Level Architecture

Acme Corp runs a microservices architecture on AWS. The system consists of
15 core services organized into 3 domains: User-Facing, Business Logic,
and Data Platform.

## Service Domains

### User-Facing Services

- **Web Gateway** — serves the React SPA, handles static assets
- **API Gateway** — Kong-based, handles routing, auth, rate limiting
- **WebSocket Service** — real-time notifications and live updates

### Business Logic Services

- **User Service** — authentication, profiles, permissions
- **Order Service** — order lifecycle, payments, fulfillment
- **Product Service** — catalog, inventory, pricing
- **Notification Service** — email, SMS, push notifications, Slack
- **Search Service** — Elasticsearch-based full-text search

### Data Platform

- **Analytics Pipeline** — Kafka + Spark, processes 50M events/day
- **Data Warehouse** — Snowflake, refreshed hourly
- **ML Platform** — SageMaker for model training and serving
- **Feature Store** — Redis-based, serves real-time ML features

## Infrastructure

### Compute

- EKS (Kubernetes) for all services
- Auto-scaling based on CPU/memory and custom request metrics
- Multi-AZ deployment in us-west-2

### Data Stores

| Service | Primary DB | Cache |
|---------|-----------|-------|
| User Service | PostgreSQL (RDS) | Redis |
| Order Service | PostgreSQL (RDS) | Redis |
| Product Service | MongoDB (DocumentDB) | Redis |
| Search Service | Elasticsearch | — |
| Analytics | Snowflake | — |

### Networking

- VPC with public, private, and data subnets
- ALB for external traffic, NLB for internal service-to-service
- CloudFront CDN for static assets (global edge locations)
- VPN for developer access to internal services

## Monitoring and Observability

### Metrics

- Datadog for infrastructure and application metrics
- Custom dashboards per service team
- SLO tracking with error budgets

### Logging

- Structured JSON logging (all services)
- Centralized in Datadog Logs (30-day retention)
- Archived to S3 (1-year retention)

### Tracing

- OpenTelemetry for distributed tracing
- Datadog APM for trace visualization
- Trace sampling: 10% of requests, 100% of errors

### Alerting

- PagerDuty for P0/P1 alerts
- Slack for P2/P3 alerts
- Alert fatigue review: monthly (aim for <5 pages/week per team)
""",
    },
    {
        "page_id": "wiki-007",
        "filename": "expense-policy.md",
        "metadata": {
            "title": "Travel and Expense Policy",
            "author": "Rachel Green",
            "last_updated": "2025-07-01",
            "department": "Finance",
            "tags": ["expenses", "travel", "reimbursement", "budget"],
        },
        "content": """# Travel and Expense Policy

## General Rules

- All expenses must be submitted within 30 days of the expense date
- Receipts required for all expenses over $25
- Manager approval required before booking travel
- Use the Expensify app for all expense submissions

## Travel

### Flights

- Book economy class for domestic flights
- Business class allowed for international flights over 6 hours
- Book at least 14 days in advance when possible (save 30-40%)
- Preferred airline: United (corporate discount code ACME2026)

### Hotels

- Maximum rate: $250/night domestic, $350/night international
- Book via TripActions (corporate rates applied automatically)
- Exceptions for high-cost cities (SF, NYC, London): up to $400/night
- Room service is not reimbursable

### Ground Transportation

- Uber/Lyft for airport transfers and local travel during trips
- Rental cars only when public transit or rideshare is impractical
- Mileage for personal vehicle: $0.67/mile (2026 IRS rate)

## Meals and Entertainment

### Team Meals

- Monthly team dinner budget: $50/person
- Manager approval required for team meals over $500 total
- Alcohol limited to 2 drinks per person (company policy)

### Client Entertainment

- Pre-approval required for client meals over $200
- Maximum per person: $100 for lunch, $150 for dinner
- Document attendees and business purpose on the receipt

### Daily Meal Per Diem (Travel)

| Location | Breakfast | Lunch | Dinner |
|----------|-----------|-------|--------|
| Domestic | $15 | $20 | $35 |
| International | $20 | $30 | $50 |

## Professional Development

- Conference attendance: manager approval + VP approval if >$2,000
- $1,500/year learning budget (courses, books, subscriptions)
- Annual professional membership dues are reimbursable

## Expense Report Process

1. Submit expense in Expensify with receipt photos
2. Categorize correctly (Travel, Meals, Equipment, etc.)
3. Manager reviews and approves (within 5 business days)
4. Finance processes approved expenses (weekly batch)
5. Reimbursement deposited in next paycheck cycle

## Non-Reimbursable Expenses

- Personal travel extensions
- Spouse/partner travel costs
- Fines, tickets, or penalties
- Personal grooming or clothing
- First-class upgrades (unless medically required)
- Mini-bar charges
""",
    },
    {
        "page_id": "wiki-008",
        "filename": "code-review-guidelines.md",
        "metadata": {
            "title": "Code Review Guidelines",
            "author": "Alex Rivera",
            "last_updated": "2026-03-01",
            "department": "Engineering",
            "tags": ["code-review", "PR", "quality", "best-practices"],
        },
        "content": """# Code Review Guidelines

## Why Code Reviews Matter

Code reviews are one of our most important quality practices. They serve to:

1. Catch bugs before they reach production
2. Share knowledge across the team
3. Maintain consistent code quality and style
4. Mentor junior developers
5. Document design decisions

## Review Requirements

### For All PRs

- Minimum 2 approvals required to merge
- All CI checks must pass (tests, lint, type check, security scan)
- No unresolved comments (resolved = addressed or acknowledged)
- PR description explains WHAT and WHY (not just HOW)

### For Critical Changes

Changes to the following require additional review:
- Authentication/authorization: Security team review
- Database migrations: DBA review
- API contracts: API guild review
- Infrastructure/Terraform: Platform team review

## How to Write a Good PR

### PR Size

- Aim for <400 lines of changes (excluding generated code)
- If the change is >400 lines, split into multiple PRs
- Each PR should be independently mergeable and testable
- "Stacked PRs" are acceptable for large features

### PR Description Template

```
## What
Brief description of the change

## Why
Motivation and context (link to ticket/issue)

## How
Key implementation decisions

## Testing
How was this tested?

## Screenshots (if UI change)
Before/after screenshots
```

## How to Review Code

### The Review Checklist

1. **Correctness** — Does the code do what it's supposed to?
2. **Design** — Is this the right approach? Are there simpler alternatives?
3. **Readability** — Can you understand the code without explanation?
4. **Tests** — Are edge cases covered? Are tests meaningful?
5. **Security** — Any injection risks, auth bypasses, data leaks?
6. **Performance** — Any N+1 queries, unbounded loops, missing indexes?

### Review Etiquette

- Review within 1 business day (4 hours ideal)
- Use "suggestion" comments for optional improvements
- Use "request changes" only for blocking issues
- Be kind: comment on the code, not the person
- Acknowledge good work ("Nice refactor!" is valuable feedback)
- Ask questions instead of making demands ("Could we...?" vs "You must...")

### Comment Prefixes

Use these prefixes to signal intent:
- `nit:` — Nitpick, optional style preference
- `suggestion:` — Non-blocking improvement idea
- `question:` — Seeking understanding, not blocking
- `concern:` — Potential issue worth discussing
- `blocker:` — Must be addressed before merge

## Response Time SLA

| PR Size | Expected Review Time |
|---------|---------------------|
| Small (<100 lines) | 4 hours |
| Medium (100-400 lines) | 1 business day |
| Large (400+ lines) | 2 business days |
| Critical/security | Same day |
""",
    },
]

print(f"Created {len(WIKI_PAGES)} wiki pages")
for page in WIKI_PAGES:
    m = page["metadata"]
    word_count = len(page["content"].split())
    print(f"  [{page['page_id']}] {m['title']}")
    print(f"    Dept: {m['department']} | Author: {m['author']} | "
          f"Updated: {m['last_updated']} | Words: {word_count}")
    print(f"    Tags: {', '.join(m['tags'])}")


Created 8 wiki pages
  [wiki-001] Production Deployment Guide
    Dept: Engineering | Author: Sarah Chen | Updated: 2026-03-15 | Words: 317
    Tags: deployment, production, CI/CD, rollback
  [wiki-002] New Employee Onboarding Guide
    Dept: HR | Author: Mike Johnson | Updated: 2026-01-10 | Words: 448
    Tags: onboarding, new-hire, setup, orientation
  [wiki-003] API Design Standards
    Dept: Engineering | Author: Priya Patel | Updated: 2025-11-20 | Words: 356
    Tags: API, REST, standards, design, backend
  [wiki-004] Incident Response Playbook
    Dept: Engineering | Author: David Kim | Updated: 2026-02-28 | Words: 396
    Tags: incident, on-call, postmortem, SLA, alerting
  [wiki-005] Data Privacy and Security Policy
    Dept: Legal | Author: Lisa Wang | Updated: 2026-04-01 | Words: 421
    Tags: privacy, security, GDPR, PII, compliance
  [wiki-006] System Architecture Overview
    Dept: Engineering | Author: James Taylor | Updated: 2025-09-15 | Words: 354
    Tags: architecture

## Markdown Parsing and Section-Level Chunking

The key to good wiki search is **heading-aware chunking**. Instead of
splitting on a fixed character count (which may cut mid-sentence or
mid-section), we split at heading boundaries.

Each section becomes a chunk with:
- Its own text content
- Its **heading path** (breadcrumb): `"Page Title > Section > Subsection"`
- The document's metadata (author, date, tags)
- Its heading depth (H1=1, H2=2, H3=3)

In [4]:
# ── Markdown section parser ──

class WikiSection:
    """A single section from a wiki page."""

    def __init__(self, page_id, heading_path, heading_depth, content,
                 page_metadata):
        self.page_id = page_id
        self.heading_path = heading_path   # e.g. "Deploy Guide > Rollback"
        self.heading_depth = heading_depth  # 1, 2, or 3
        self.content = content.strip()
        self.page_metadata = page_metadata

        # Search text = heading path + content
        self.search_text = f"{heading_path}. {self.content}"

        # Unique chunk ID
        text_hash = hashlib.md5(self.search_text.encode()).hexdigest()[:8]
        self.chunk_id = f"{page_id}-{text_hash}"

    @property
    def breadcrumb(self):
        return self.heading_path

    @property
    def summary(self):
        preview = self.content[:80].replace("\n", " ")
        return f"[{self.chunk_id}] {self.breadcrumb} | {preview}..."


def parse_wiki_page(page):
    """Parse a markdown wiki page into heading-aware sections."""
    content = page["content"]
    metadata = page["metadata"]
    page_id = page["page_id"]

    lines = content.split("\n")
    sections = []
    heading_stack = []  # [(depth, heading_text), ...]
    current_content_lines = []

    def flush_section():
        text = "\n".join(current_content_lines).strip()
        if text and heading_stack:
            path = " > ".join(h for _, h in heading_stack)
            depth = heading_stack[-1][0]
            sections.append(WikiSection(
                page_id=page_id,
                heading_path=path,
                heading_depth=depth,
                content=text,
                page_metadata=metadata,
            ))

    for line in lines:
        # Detect heading level
        heading_match = re.match(r"^(#{1,3})\s+(.+)$", line)
        if heading_match:
            # Flush the previous section
            flush_section()
            current_content_lines = []

            depth = len(heading_match.group(1))
            heading_text = heading_match.group(2).strip()

            # Pop headings at same or deeper level
            while heading_stack and heading_stack[-1][0] >= depth:
                heading_stack.pop()
            heading_stack.append((depth, heading_text))
        else:
            current_content_lines.append(line)

    # Flush last section
    flush_section()

    return sections


# Parse all wiki pages
all_sections = []
for page in WIKI_PAGES:
    page_sections = parse_wiki_page(page)
    all_sections.extend(page_sections)
    print(f"  {page['metadata']['title']}: {len(page_sections)} sections")

print(f"\nTotal sections: {len(all_sections)}")

  Production Deployment Guide: 8 sections
  New Employee Onboarding Guide: 11 sections
  API Design Standards: 10 sections
  Incident Response Playbook: 13 sections
  Data Privacy and Security Policy: 12 sections
  System Architecture Overview: 11 sections
  Travel and Expense Policy: 10 sections
  Code Review Guidelines: 14 sections

Total sections: 89


In [5]:
# ── Inspect parsed sections ──

print("=== Sample Parsed Sections ===\n")
for section in all_sections[:8]:
    print(f"  Breadcrumb: {section.breadcrumb}")
    print(f"  Depth: H{section.heading_depth} | "
          f"Dept: {section.page_metadata['department']} | "
          f"Tags: {section.page_metadata['tags'][:3]}")
    preview = section.content[:100].replace("\n", " ")
    print(f"  Content: {preview}...")
    print()

# Stats
depths = Counter(s.heading_depth for s in all_sections)
depts = Counter(s.page_metadata["department"] for s in all_sections)
avg_len = np.mean([len(s.content) for s in all_sections])

print(f"Heading depths: {dict(depths)}")
print(f"Departments: {dict(depts)}")
print(f"Avg section length: {avg_len:.0f} chars")

=== Sample Parsed Sections ===

  Breadcrumb: Production Deployment Guide > Overview
  Depth: H2 | Dept: Engineering | Tags: ['deployment', 'production', 'CI/CD']
  Content: This guide covers Acme Corp's production deployment process. All deployments follow our CI/CD pipeli...

  Breadcrumb: Production Deployment Guide > Pre-Deployment Checklist
  Depth: H2 | Dept: Engineering | Tags: ['deployment', 'production', 'CI/CD']
  Content: Before initiating a deployment:  1. All tests pass on the main branch (unit, integration, e2e) 2. Co...

  Breadcrumb: Production Deployment Guide > Deployment Steps > Step 1: Create Release Branch
  Depth: H3 | Dept: Engineering | Tags: ['deployment', 'production', 'CI/CD']
  Content: Create a release branch from main: ``` git checkout -b release/v2.5.0 main ```  Tag the release and ...

  Breadcrumb: Production Deployment Guide > Deployment Steps > Step 2: Deploy to Staging
  Depth: H3 | Dept: Engineering | Tags: ['deployment', 'production', 'CI/CD']
  Co

## Building the Section-Level Search Index

In [6]:
# ── Wiki search index ──

class WikiIndex:
    """Section-level search index over wiki pages."""

    def __init__(self, sections):
        self.sections = sections
        self.texts = [s.search_text for s in sections]
        self.section_map = {s.chunk_id: s for s in sections}

        if USE_ST and USE_CHROMA:
            self._build_chroma()
        else:
            self._build_tfidf()

    def _build_chroma(self):
        print("Building index with sentence-transformers + ChromaDB...")
        self.model = SentenceTransformer(EMBEDDING_MODEL)
        self.client = chromadb.Client()
        try:
            self.client.delete_collection("wiki")
        except Exception:
            pass
        self.collection = self.client.create_collection(
            name="wiki", metadata={"hnsw:space": "cosine"}
        )
        embs = self.model.encode(self.texts, show_progress_bar=False).tolist()
        ids = [s.chunk_id for s in self.sections]
        metas = [{
            "page_id": s.page_id,
            "department": s.page_metadata["department"],
            "heading_depth": str(s.heading_depth),
        } for s in self.sections]
        self.collection.add(ids=ids, embeddings=embs,
                            documents=self.texts, metadatas=metas)
        self.backend = "chroma"
        print(f"Indexed {self.collection.count()} sections")

    def _build_tfidf(self):
        print("Building TF-IDF index...")
        self.vectorizer = TfidfVectorizer(
            max_features=5000, stop_words="english", ngram_range=(1, 2)
        )
        self.tfidf_matrix = self.vectorizer.fit_transform(self.texts)
        self.backend = "tfidf"
        print(f"Indexed {self.tfidf_matrix.shape[0]} sections (TF-IDF)")

    def search(self, query, top_k=TOP_K, department=None):
        if self.backend == "chroma":
            return self._search_chroma(query, top_k, department)
        return self._search_tfidf(query, top_k, department)

    def _search_chroma(self, query, top_k, department):
        where = {"department": department} if department else None
        query_emb = self.model.encode([query]).tolist()
        n = min(top_k * 2, self.collection.count())
        results = self.collection.query(
            query_embeddings=query_emb, n_results=n, where=where
        )
        output = []
        for i, cid in enumerate(results["ids"][0]):
            section = self.section_map[cid]
            sim = 1.0 - results["distances"][0][i]
            output.append((section, sim))
        return output[:top_k]

    def _search_tfidf(self, query, top_k, department):
        candidates = []
        for i, s in enumerate(self.sections):
            if department and s.page_metadata["department"] != department:
                continue
            candidates.append(i)
        if not candidates:
            return []
        query_vec = self.vectorizer.transform([query])
        cand_matrix = self.tfidf_matrix[candidates]
        sims = cosine_similarity(query_vec, cand_matrix).flatten()
        top_idx = np.argsort(sims)[::-1][:top_k]
        return [(self.sections[candidates[i]], float(sims[i])) for i in top_idx]


wiki_index = WikiIndex(all_sections)

Building index with sentence-transformers + ChromaDB...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Indexed 89 sections


## Metadata-Boosted Retrieval

Raw semantic similarity is a good starting point, but we can improve
ranking by incorporating document metadata:

1. **Recency boost** — recently updated docs are more likely to be current
2. **Tag match boost** — if the query matches a document tag, boost that result
3. **Heading depth penalty** — top-level sections (H1, H2) are slightly
   preferred over deeply nested ones (H3) for broad queries

In [7]:
# ── Metadata boosted search ──

def metadata_boosted_search(index, query, top_k=TOP_K, department=None):
    """Combine semantic similarity with metadata signals."""

    raw_results = index.search(query, top_k=top_k * 2, department=department)
    query_lower = query.lower()

    boosted = []
    for section, sim_score in raw_results:
        boost = 0.0

        # 1. Recency boost
        try:
            updated = datetime.strptime(
                section.page_metadata["last_updated"], "%Y-%m-%d"
            )
            years_ago = (datetime(2026, 4, 16) - updated).days / 365.25
            recency_score = max(0, (5 - years_ago) * RECENCY_BOOST)
            boost += recency_score
        except (ValueError, KeyError):
            pass

        # 2. Tag match boost
        tags = section.page_metadata.get("tags", [])
        for tag in tags:
            if tag.lower() in query_lower or query_lower in tag.lower():
                boost += TAG_MATCH_BOOST
                break  # max one tag boost per section

        # 3. Heading depth penalty (slight preference for top-level)
        depth_penalty = (section.heading_depth - 1) * HEADING_DEPTH_PENALTY
        boost -= depth_penalty

        final_score = sim_score + boost
        boosted.append((section, sim_score, boost, final_score))

    boosted.sort(key=lambda x: x[3], reverse=True)
    return boosted[:top_k]


# Test
test_q = "how to deploy to production"
results = metadata_boosted_search(wiki_index, test_q, top_k=3)

print(f"Query: \"{test_q}\"\n")
for section, sim, boost, final in results:
    print(f"  sim={sim:.3f} boost={boost:+.3f} final={final:.3f}")
    print(f"  Breadcrumb: {section.breadcrumb}")
    print(f"  Dept: {section.page_metadata['department']} | "
          f"Updated: {section.page_metadata['last_updated']}")
    print()

Query: "how to deploy to production"

  sim=0.672 boost=+0.306 final=0.978
  Breadcrumb: Production Deployment Guide > Deployment Steps > Step 3: Production Promotion
  Dept: Engineering | Updated: 2026-03-15

  sim=0.664 boost=+0.306 final=0.970
  Breadcrumb: Production Deployment Guide > Deployment Steps > Step 2: Deploy to Staging
  Dept: Engineering | Updated: 2026-03-15

  sim=0.617 boost=+0.326 final=0.943
  Breadcrumb: Production Deployment Guide > Emergency Deployment Process
  Dept: Engineering | Updated: 2026-03-15



## Baseline: Keyword Search

In [8]:
# ── Keyword search baseline ──

def keyword_search(query, sections, top_k=TOP_K):
    q_terms = set(query.lower().split())
    scored = []
    for s in sections:
        text_lower = s.search_text.lower()
        hits = sum(1 for t in q_terms if t in text_lower)
        scored.append((s, hits / max(len(q_terms), 1)))
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]

test_q = "rollback a failed release"
kw = keyword_search(test_q, all_sections, top_k=3)

print(f"Query: \"{test_q}\"\n")
print("Keyword results:")
for s, score in kw:
    print(f"  {score:.2f} | {s.breadcrumb}")

Query: "rollback a failed release"

Keyword results:
  0.50 | Production Deployment Guide > Pre-Deployment Checklist
  0.50 | Production Deployment Guide > Deployment Steps > Step 1: Create Release Branch
  0.50 | Production Deployment Guide > Deployment Steps > Step 2: Deploy to Staging


## Internal Wiki Assistant Pipeline

In [9]:
# ── Wiki Assistant ──

class WikiAssistant:
    """Search the internal wiki with metadata-boosted retrieval."""

    def __init__(self, index, sections):
        self.index = index
        self.sections = sections

    def ask(self, question, department=None, top_k=TOP_K, verbose=True):
        """Ask a question and get the most relevant wiki sections."""
        results = metadata_boosted_search(
            self.index, question, top_k=top_k, department=department
        )

        # Filter by threshold
        relevant = [(s, sim, boost, final)
                     for s, sim, boost, final in results
                     if final >= SIMILARITY_THRESHOLD]

        output = {
            "question": question,
            "department_filter": department,
            "results_count": len(relevant),
            "results": [{
                "breadcrumb": s.breadcrumb,
                "page_title": s.page_metadata["title"],
                "department": s.page_metadata["department"],
                "author": s.page_metadata["author"],
                "last_updated": s.page_metadata["last_updated"],
                "tags": s.page_metadata["tags"],
                "similarity": round(sim, 3),
                "metadata_boost": round(boost, 3),
                "final_score": round(final, 3),
                "content": s.content,
            } for s, sim, boost, final in relevant[:top_k]],
        }

        if verbose:
            self._display(output)
        return output

    def _display(self, output):
        print("=" * 72)
        print(f"Q: {output['question']}")
        if output["department_filter"]:
            print(f"Filter: department={output['department_filter']}")
        print("=" * 72)

        if not output["results"]:
            print("  No relevant wiki sections found.")
            return

        for i, r in enumerate(output["results"][:3], 1):
            print(f"\n  [{i}] {r['breadcrumb']}")
            print(f"      Page: {r['page_title']}")
            print(f"      Dept: {r['department']} | Author: {r['author']} | "
                  f"Updated: {r['last_updated']}")
            print(f"      Score: sim={r['similarity']:.3f} "
                  f"boost={r['metadata_boost']:+.3f} "
                  f"final={r['final_score']:.3f}")

            content_preview = r["content"][:200].replace("\n", " ")
            wrapped = textwrap.fill(content_preview,
                                     width=66,
                                     initial_indent="      ",
                                     subsequent_indent="      ")
            print(wrapped + "...")

        print("=" * 72)


assistant = WikiAssistant(wiki_index, all_sections)
print("Wiki Assistant ready.")


Wiki Assistant ready.


## Query Demonstrations

In [10]:
# ── Demo 1: Deployment question ──
assistant.ask("How do I roll back a production deployment?")

Q: How do I roll back a production deployment?

  [1] Production Deployment Guide > Rollback Procedure
      Page: Production Deployment Guide
      Dept: Engineering | Author: Sarah Chen | Updated: 2026-03-15
      Score: sim=0.708 boost=+0.326 final=1.034
      If issues are detected after deployment:  1. Go to the
      deployment dashboard 2. Click "Rollback" on the current
      deployment 3. Select the previous stable version 4. Confirm
      rollback (takes ~5 minutes) 5...

  [2] Production Deployment Guide > Pre-Deployment Checklist
      Page: Production Deployment Guide
      Dept: Engineering | Author: Sarah Chen | Updated: 2026-03-15
      Score: sim=0.564 boost=+0.326 final=0.889
      Before initiating a deployment:  1. All tests pass on the
      main branch (unit, integration, e2e) 2. Code review approved
      by at least 2 reviewers 3. Changelog updated with version
      number 4. Feature flags...

  [3] Production Deployment Guide > Emergency Deployment Process
   

{'question': 'How do I roll back a production deployment?',
 'department_filter': None,
 'results_count': 5,
 'results': [{'breadcrumb': 'Production Deployment Guide > Rollback Procedure',
   'page_title': 'Production Deployment Guide',
   'department': 'Engineering',
   'author': 'Sarah Chen',
   'last_updated': '2026-03-15',
   'tags': ['deployment', 'production', 'CI/CD', 'rollback'],
   'similarity': 0.708,
   'metadata_boost': 0.326,
   'final_score': 1.034,
   'content': 'If issues are detected after deployment:\n\n1. Go to the deployment dashboard\n2. Click "Rollback" on the current deployment\n3. Select the previous stable version\n4. Confirm rollback (takes ~5 minutes)\n5. Notify the team in #engineering-alerts\n\nRollbacks are instant for configuration changes but take 5-10 minutes\nfor code deployments due to container restart time.'},
  {'breadcrumb': 'Production Deployment Guide > Pre-Deployment Checklist',
   'page_title': 'Production Deployment Guide',
   'department': '

In [11]:
# ── Demo 2: HR question ──
assistant.ask("What is the PTO policy and how many vacation days do I get?")

Q: What is the PTO policy and how many vacation days do I get?

  [1] New Employee Onboarding Guide > Benefits Overview > Time Off
      Page: New Employee Onboarding Guide
      Dept: HR | Author: Mike Johnson | Updated: 2026-01-10
      Score: sim=0.615 boost=+0.197 final=0.812
      - 20 days PTO per year (accrued monthly) - 10 company
      holidays - 5 sick days (separate from PTO) - Parental leave:
      16 weeks paid...

  [2] Travel and Expense Policy > General Rules
      Page: Travel and Expense Policy
      Dept: Finance | Author: Rachel Green | Updated: 2025-07-01
      Score: sim=0.449 boost=+0.190 final=0.639
      - All expenses must be submitted within 30 days of the
      expense date - Receipts required for all expenses over $25 -
      Manager approval required before booking travel - Use the
      Expensify app for all expe...

  [3] New Employee Onboarding Guide > Benefits Overview > Health Insurance
      Page: New Employee Onboarding Guide
      Dept: HR | Author

{'question': 'What is the PTO policy and how many vacation days do I get?',
 'department_filter': None,
 'results_count': 5,
 'results': [{'breadcrumb': 'New Employee Onboarding Guide > Benefits Overview > Time Off',
   'page_title': 'New Employee Onboarding Guide',
   'department': 'HR',
   'author': 'Mike Johnson',
   'last_updated': '2026-01-10',
   'tags': ['onboarding', 'new-hire', 'setup', 'orientation'],
   'similarity': 0.615,
   'metadata_boost': 0.197,
   'final_score': 0.812,
   'content': '- 20 days PTO per year (accrued monthly)\n- 10 company holidays\n- 5 sick days (separate from PTO)\n- Parental leave: 16 weeks paid'},
  {'breadcrumb': 'Travel and Expense Policy > General Rules',
   'page_title': 'Travel and Expense Policy',
   'department': 'Finance',
   'author': 'Rachel Green',
   'last_updated': '2025-07-01',
   'tags': ['expenses', 'travel', 'reimbursement', 'budget'],
   'similarity': 0.449,
   'metadata_boost': 0.19,
   'final_score': 0.639,
   'content': '- All e

In [12]:
# ── Demo 3: API standards ──
assistant.ask("What HTTP status codes should our API return?")

Q: What HTTP status codes should our API return?

  [1] API Design Standards > Request and Response Format > Response Codes
      Page: API Design Standards
      Dept: Engineering | Author: Priya Patel | Updated: 2025-11-20
      Score: sim=0.653 boost=+0.290 final=0.943
      - 200: Success (GET, PUT, PATCH) - 201: Created (POST) -
      204: No Content (DELETE) - 400: Bad Request (validation
      error) - 401: Unauthorized (missing/invalid auth) - 403:
      Forbidden (insufficient permis...

  [2] API Design Standards > Request and Response Format > HTTP Methods
      Page: API Design Standards
      Dept: Engineering | Author: Priya Patel | Updated: 2025-11-20
      Score: sim=0.604 boost=+0.290 final=0.894
      | Method | Usage | Idempotent | |--------|-------|:---:| |
      GET | Retrieve resource(s) | Yes | | POST | Create new
      resource | No | | PUT | Full resource update | Yes | | PATCH
      | Partial update | No | |...

  [3] API Design Standards > Purpose
      Page:

{'question': 'What HTTP status codes should our API return?',
 'department_filter': None,
 'results_count': 5,
 'results': [{'breadcrumb': 'API Design Standards > Request and Response Format > Response Codes',
   'page_title': 'API Design Standards',
   'department': 'Engineering',
   'author': 'Priya Patel',
   'last_updated': '2025-11-20',
   'tags': ['API', 'REST', 'standards', 'design', 'backend'],
   'similarity': 0.653,
   'metadata_boost': 0.29,
   'final_score': 0.943,
   'content': '- 200: Success (GET, PUT, PATCH)\n- 201: Created (POST)\n- 204: No Content (DELETE)\n- 400: Bad Request (validation error)\n- 401: Unauthorized (missing/invalid auth)\n- 403: Forbidden (insufficient permissions)\n- 404: Not Found\n- 409: Conflict (duplicate resource)\n- 422: Unprocessable Entity (semantic error)\n- 429: Too Many Requests (rate limited)\n- 500: Internal Server Error'},
  {'breadcrumb': 'API Design Standards > Request and Response Format > HTTP Methods',
   'page_title': 'API Design 

In [13]:
# ── Demo 4: Incident response ──
assistant.ask("What do I do during a P0 production outage?")

Q: What do I do during a P0 production outage?

  [1] Production Deployment Guide > Emergency Deployment Process
      Page: Production Deployment Guide
      Dept: Engineering | Author: Sarah Chen | Updated: 2026-03-15
      Score: sim=0.607 boost=+0.326 final=0.933
      For critical production issues (P0/P1):  1. Get verbal
      approval from VP Engineering or CTO 2. Create hotfix branch
      from the last stable release tag 3. Minimum 1 code review
      (can be done in real-time)...

  [2] Production Deployment Guide > Deployment Windows
      Page: Production Deployment Guide
      Dept: Engineering | Author: Sarah Chen | Updated: 2026-03-15
      Score: sim=0.409 boost=+0.326 final=0.734
      - **Standard deployments:** Tuesday-Thursday, 10am-4pm PT -
      **Hotfixes:** Any time, but require VP Engineering approval
      outside business hours - **Database migrations:** Wednesday
      2-4pm PT only (DBA m...

  [3] Production Deployment Guide > Rollback Procedure
      Page

{'question': 'What do I do during a P0 production outage?',
 'department_filter': None,
 'results_count': 5,
 'results': [{'breadcrumb': 'Production Deployment Guide > Emergency Deployment Process',
   'page_title': 'Production Deployment Guide',
   'department': 'Engineering',
   'author': 'Sarah Chen',
   'last_updated': '2026-03-15',
   'tags': ['deployment', 'production', 'CI/CD', 'rollback'],
   'similarity': 0.607,
   'metadata_boost': 0.326,
   'final_score': 0.933,
   'content': 'For critical production issues (P0/P1):\n\n1. Get verbal approval from VP Engineering or CTO\n2. Create hotfix branch from the last stable release tag\n3. Minimum 1 code review (can be done in real-time)\n4. Deploy directly to production (skip staging soak time)\n5. File incident report within 24 hours'},
  {'breadcrumb': 'Production Deployment Guide > Deployment Windows',
   'page_title': 'Production Deployment Guide',
   'department': 'Engineering',
   'author': 'Sarah Chen',
   'last_updated': '2026

In [14]:
# ── Demo 5: Data privacy ──
assistant.ask("How do we handle GDPR data deletion requests?")

Q: How do we handle GDPR data deletion requests?

  [1] Data Privacy and Security Policy > Personal Data Handling (GDPR/CCPA) > Data Processing Rules
      Page: Data Privacy and Security Policy
      Dept: Legal | Author: Lisa Wang | Updated: 2026-04-01
      Score: sim=0.536 boost=+0.308 final=0.844
      1. Collect only what is necessary (data minimization) 2.
      Explain data use in plain language (transparency) 3. Get
      explicit consent before collecting (consent) 4. Allow users
      to export their data (port...

  [2] Data Privacy and Security Policy > Personal Data Handling (GDPR/CCPA) > Data Retention
      Page: Data Privacy and Security Policy
      Dept: Legal | Author: Lisa Wang | Updated: 2026-04-01
      Score: sim=0.536 boost=+0.308 final=0.844
      | Data Type | Retention Period | After Retention |
      |-----------|-----------------|-----------------| | Account
      data | Duration of account + 30 days | Deleted | | Usage
      logs | 12 months | Anonymized 

{'question': 'How do we handle GDPR data deletion requests?',
 'department_filter': None,
 'results_count': 5,
 'results': [{'breadcrumb': 'Data Privacy and Security Policy > Personal Data Handling (GDPR/CCPA) > Data Processing Rules',
   'page_title': 'Data Privacy and Security Policy',
   'department': 'Legal',
   'author': 'Lisa Wang',
   'last_updated': '2026-04-01',
   'tags': ['privacy', 'security', 'GDPR', 'PII', 'compliance'],
   'similarity': 0.536,
   'metadata_boost': 0.308,
   'final_score': 0.844,
   'content': '1. Collect only what is necessary (data minimization)\n2. Explain data use in plain language (transparency)\n3. Get explicit consent before collecting (consent)\n4. Allow users to export their data (portability)\n5. Allow users to request deletion (right to erasure)\n6. Notify within 72 hours of a breach (breach notification)'},
  {'breadcrumb': 'Data Privacy and Security Policy > Personal Data Handling (GDPR/CCPA) > Data Retention',
   'page_title': 'Data Privacy 

In [15]:
# ── Demo 6: Department-filtered search ──
assistant.ask("What is our expense reimbursement process?",
              department="Finance")


Q: What is our expense reimbursement process?
Filter: department=Finance

  [1] Travel and Expense Policy > Expense Report Process
      Page: Travel and Expense Policy
      Dept: Finance | Author: Rachel Green | Updated: 2025-07-01
      Score: sim=0.607 boost=+0.290 final=0.897
      1. Submit expense in Expensify with receipt photos 2.
      Categorize correctly (Travel, Meals, Equipment, etc.) 3.
      Manager reviews and approves (within 5 business days) 4.
      Finance processes approved expen...

  [2] Travel and Expense Policy > General Rules
      Page: Travel and Expense Policy
      Dept: Finance | Author: Rachel Green | Updated: 2025-07-01
      Score: sim=0.473 boost=+0.290 final=0.764
      - All expenses must be submitted within 30 days of the
      expense date - Receipts required for all expenses over $25 -
      Manager approval required before booking travel - Use the
      Expensify app for all expe...

  [3] Travel and Expense Policy > Non-Reimbursable Expenses


{'question': 'What is our expense reimbursement process?',
 'department_filter': 'Finance',
 'results_count': 5,
 'results': [{'breadcrumb': 'Travel and Expense Policy > Expense Report Process',
   'page_title': 'Travel and Expense Policy',
   'department': 'Finance',
   'author': 'Rachel Green',
   'last_updated': '2025-07-01',
   'tags': ['expenses', 'travel', 'reimbursement', 'budget'],
   'similarity': 0.607,
   'metadata_boost': 0.29,
   'final_score': 0.897,
   'content': '1. Submit expense in Expensify with receipt photos\n2. Categorize correctly (Travel, Meals, Equipment, etc.)\n3. Manager reviews and approves (within 5 business days)\n4. Finance processes approved expenses (weekly batch)\n5. Reimbursement deposited in next paycheck cycle'},
  {'breadcrumb': 'Travel and Expense Policy > General Rules',
   'page_title': 'Travel and Expense Policy',
   'department': 'Finance',
   'author': 'Rachel Green',
   'last_updated': '2025-07-01',
   'tags': ['expenses', 'travel', 'reimbur

In [16]:
# ── Demo 7: Code review process ──
assistant.ask("How should I write a good pull request description?")

Q: How should I write a good pull request description?

  [1] Code Review Guidelines > How to Write a Good PR > PR Description Template
      Page: Code Review Guidelines
      Dept: Engineering | Author: Alex Rivera | Updated: 2026-03-01
      Score: sim=0.467 boost=+0.204 final=0.670
      ```...

  [2] Code Review Guidelines > What
      Page: Code Review Guidelines
      Dept: Engineering | Author: Alex Rivera | Updated: 2026-03-01
      Score: sim=0.364 boost=+0.224 final=0.587
      Brief description of the change...

  [3] API Design Standards > Request and Response Format > HTTP Methods
      Page: API Design Standards
      Dept: Engineering | Author: Priya Patel | Updated: 2025-11-20
      Score: sim=0.361 boost=+0.190 final=0.551
      | Method | Usage | Idempotent | |--------|-------|:---:| |
      GET | Retrieve resource(s) | Yes | | POST | Create new
      resource | No | | PUT | Full resource update | Yes | | PATCH
      | Partial update | No | |...


{'question': 'How should I write a good pull request description?',
 'department_filter': None,
 'results_count': 5,
 'results': [{'breadcrumb': 'Code Review Guidelines > How to Write a Good PR > PR Description Template',
   'page_title': 'Code Review Guidelines',
   'department': 'Engineering',
   'author': 'Alex Rivera',
   'last_updated': '2026-03-01',
   'tags': ['code-review', 'PR', 'quality', 'best-practices'],
   'similarity': 0.467,
   'metadata_boost': 0.204,
   'final_score': 0.67,
   'content': '```'},
  {'breadcrumb': 'Code Review Guidelines > What',
   'page_title': 'Code Review Guidelines',
   'department': 'Engineering',
   'author': 'Alex Rivera',
   'last_updated': '2026-03-01',
   'tags': ['code-review', 'PR', 'quality', 'best-practices'],
   'similarity': 0.364,
   'metadata_boost': 0.224,
   'final_score': 0.587,
   'content': 'Brief description of the change'},
  {'breadcrumb': 'API Design Standards > Request and Response Format > HTTP Methods',
   'page_title': 'A

In [17]:
# ── Demo 8: Cross-department discovery ──
assistant.ask("What monitoring and alerting tools do we use?")

Q: What monitoring and alerting tools do we use?

  [1] System Architecture Overview > Monitoring and Observability > Alerting
      Page: System Architecture Overview
      Dept: Engineering | Author: James Taylor | Updated: 2025-09-15
      Score: sim=0.687 boost=+0.181 final=0.868
      - PagerDuty for P0/P1 alerts - Slack for P2/P3 alerts -
      Alert fatigue review: monthly (aim for <5 pages/week per
      team)...

  [2] Incident Response Playbook > On-Call Responsibilities > Primary On-Call
      Page: Incident Response Playbook
      Dept: Engineering | Author: David Kim | Updated: 2026-02-28
      Score: sim=0.455 boost=+0.304 final=0.758
      - Monitor alerts in PagerDuty - Acknowledge pages within 5
      minutes - Triage and assess severity - Engage secondary on-
      call if needed - Update status page for P0/P1...

  [3] System Architecture Overview > Monitoring and Observability > Metrics
      Page: System Architecture Overview
      Dept: Engineering | Author: James

{'question': 'What monitoring and alerting tools do we use?',
 'department_filter': None,
 'results_count': 5,
 'results': [{'breadcrumb': 'System Architecture Overview > Monitoring and Observability > Alerting',
   'page_title': 'System Architecture Overview',
   'department': 'Engineering',
   'author': 'James Taylor',
   'last_updated': '2025-09-15',
   'tags': ['architecture', 'microservices', 'infrastructure', 'AWS'],
   'similarity': 0.687,
   'metadata_boost': 0.181,
   'final_score': 0.868,
   'content': '- PagerDuty for P0/P1 alerts\n- Slack for P2/P3 alerts\n- Alert fatigue review: monthly (aim for <5 pages/week per team)'},
  {'breadcrumb': 'Incident Response Playbook > On-Call Responsibilities > Primary On-Call',
   'page_title': 'Incident Response Playbook',
   'department': 'Engineering',
   'author': 'David Kim',
   'last_updated': '2026-02-28',
   'tags': ['incident', 'on-call', 'postmortem', 'SLA', 'alerting'],
   'similarity': 0.455,
   'metadata_boost': 0.304,
   'fi

## Section-Level vs Page-Level Indexing

Let's demonstrate why section-level indexing produces better results than
indexing entire pages.

In [18]:
# ── Page-level index for comparison ──

class PageLevelIndex:
    """Full-page indexing for comparison."""

    def __init__(self, pages):
        self.pages = pages
        self.texts = [
            f"{p['metadata']['title']}. {p['content']}" for p in pages
        ]
        self.vectorizer = TfidfVectorizer(
            max_features=5000, stop_words="english", ngram_range=(1, 2)
        )
        self.tfidf_matrix = self.vectorizer.fit_transform(self.texts)

    def search(self, query, top_k=3):
        query_vec = self.vectorizer.transform([query])
        sims = cosine_similarity(query_vec, self.tfidf_matrix).flatten()
        top_idx = np.argsort(sims)[::-1][:top_k]
        return [(self.pages[i], float(sims[i])) for i in top_idx]


page_index = PageLevelIndex(WIKI_PAGES)

# Compare
test_queries = [
    "rollback procedure for failed deployment",
    "how to request VPN access",
    "postmortem template for incidents",
    "API rate limiting configuration",
]

print(f"{'Query':<45} {'Section Result':<35} {'Page Result':<30}")
print("-" * 110)

for q in test_queries:
    # Section-level result
    sec_results = metadata_boosted_search(wiki_index, q, top_k=1)
    sec_breadcrumb = sec_results[0][0].breadcrumb if sec_results else "N/A"

    # Page-level result
    page_results = page_index.search(q, top_k=1)
    page_title = page_results[0][0]["metadata"]["title"] if page_results else "N/A"

    print(f"  {q[:43]:<43} {sec_breadcrumb[:33]:<33} {page_title[:28]}")

print("\nSection-level results point to the EXACT section, not just the page.")
print("Page-level results only tell you which page — you still need to find the answer.")

Query                                         Section Result                      Page Result                   
--------------------------------------------------------------------------------------------------------------
  rollback procedure for failed deployment    Production Deployment Guide > Rol Production Deployment Guide


  how to request VPN access                   New Employee Onboarding Guide > A New Employee Onboarding Guid


  postmortem template for incidents           Incident Response Playbook > Post Incident Response Playbook
  API rate limiting configuration             API Design Standards > Rate Limit API Design Standards

Section-level results point to the EXACT section, not just the page.
Page-level results only tell you which page — you still need to find the answer.


## Evaluation

In [19]:
# ── Evaluation set ──

EVAL_SET = [
    {"query": "what's the rollback process for production",
     "expected_page": "wiki-001",
     "expected_breadcrumb_contains": "Rollback"},

    {"query": "first day orientation schedule for new employees",
     "expected_page": "wiki-002",
     "expected_breadcrumb_contains": "Day 1"},

    {"query": "REST API pagination standards",
     "expected_page": "wiki-003",
     "expected_breadcrumb_contains": "Pagination"},

    {"query": "incident severity levels and response times",
     "expected_page": "wiki-004",
     "expected_breadcrumb_contains": "Severity"},

    {"query": "personal data retention periods GDPR",
     "expected_page": "wiki-005",
     "expected_breadcrumb_contains": "Retention"},

    {"query": "what databases each service uses",
     "expected_page": "wiki-006",
     "expected_breadcrumb_contains": "Data Store"},

    {"query": "how much is meal per diem for international travel",
     "expected_page": "wiki-007",
     "expected_breadcrumb_contains": "Meal"},

    {"query": "code review turnaround time expectations",
     "expected_page": "wiki-008",
     "expected_breadcrumb_contains": "Response"},

    {"query": "how to create a release branch and tag",
     "expected_page": "wiki-001",
     "expected_breadcrumb_contains": "Release"},

    {"query": "employee health insurance coverage details",
     "expected_page": "wiki-002",
     "expected_breadcrumb_contains": "Health"},
]

n_eval = len(EVAL_SET)

# Evaluate: page recall and section recall
page_hits = 0
section_hits = 0
kw_page_hits = 0
mrr_values = []

print(f"{'Query':<50} {'Page':^8} {'Section':^8} {'KW Page':^8}")
print("-" * 76)

for item in EVAL_SET:
    q = item["query"]
    exp_page = item["expected_page"]
    exp_bc = item["expected_breadcrumb_contains"]

    # Metadata-boosted semantic search
    results = metadata_boosted_search(wiki_index, q, top_k=5)
    result_pages = [s.page_id for s, _, _, _ in results]
    result_bcs = [s.breadcrumb for s, _, _, _ in results]

    p_hit = exp_page in result_pages[:1]
    page_hits += int(p_hit)

    s_hit = any(exp_bc.lower() in bc.lower() for bc in result_bcs[:3])
    section_hits += int(s_hit)

    rank = (result_pages.index(exp_page) + 1) if exp_page in result_pages else 0
    mrr_values.append(1.0 / rank if rank > 0 else 0)

    # Keyword baseline
    kw = keyword_search(q, all_sections, top_k=1)
    kw_page = kw[0][0].page_id if kw else "???"
    kw_hit = kw_page == exp_page
    kw_page_hits += int(kw_hit)

    print(f"  {q[:48]:<48} {'HIT' if p_hit else 'miss':^8} "
          f"{'HIT' if s_hit else 'miss':^8} "
          f"{'HIT' if kw_hit else 'miss':^8}")

print("-" * 76)
print(f"\n{'Metric':<30} {'Boosted':^12} {'Keyword':^12}")
print("-" * 54)
print(f"  {'Page Recall@1':<28} {page_hits/n_eval:^12.1%} "
      f"{kw_page_hits/n_eval:^12.1%}")
print(f"  {'Section Recall@3':<28} {section_hits/n_eval:^12.1%} {'N/A':^12}")
print(f"  {'MRR':<28} {np.mean(mrr_values):^12.3f} {'N/A':^12}")

Query                                                Page   Section  KW Page 
----------------------------------------------------------------------------
  what's the rollback process for production         HIT      HIT      HIT   


  first day orientation schedule for new employees   HIT      HIT      HIT   
  REST API pagination standards                      HIT      HIT      HIT   
  incident severity levels and response times        HIT      HIT      HIT   


  personal data retention periods GDPR               HIT      HIT      HIT   


  what databases each service uses                   HIT      HIT      miss  
  how much is meal per diem for international trav   HIT      HIT      HIT   


  code review turnaround time expectations           HIT      HIT      miss  


  how to create a release branch and tag             HIT      HIT      HIT   


  employee health insurance coverage details         HIT      HIT      HIT   
----------------------------------------------------------------------------

Metric                           Boosted      Keyword   
------------------------------------------------------
  Page Recall@1                   100.0%       80.0%    
  Section Recall@3                100.0%        N/A     
  MRR                             1.000         N/A     


## Error Analysis

In [20]:
# ── Edge cases ──

edge_cases = [
    ("where is the office kitchen",
     "out_of_scope",
     "Facilities question — not in the wiki corpus"),
    ("tell me about quantum computing",
     "completely_off_topic",
     "Not related to any wiki content"),
    ("deploy",
     "too_short",
     "Very short query — ambiguous without context"),
    ("how does the oncall thing work and what do I do if something breaks at 3am and should I wake up the manager",
     "too_long",
     "Very verbose query — embedding may dilute signal"),
]

print("=== Edge Case Analysis ===\n")
for query, case_type, note in edge_cases:
    results = metadata_boosted_search(wiki_index, query, top_k=2)
    top_score = results[0][3] if results else 0

    print(f"Q: \"{query[:65]}\"")
    print(f"  Case: {case_type}")
    print(f"  Top final score: {top_score:.3f}")
    if results:
        print(f"  Top result: {results[0][0].breadcrumb}")
    if top_score < SIMILARITY_THRESHOLD:
        print(f"  -> Below threshold ({SIMILARITY_THRESHOLD}): no confident match")
    print(f"  Note: {note}")
    print()

=== Edge Case Analysis ===

Q: "where is the office kitchen"
  Case: out_of_scope
  Top final score: 0.451
  Top result: New Employee Onboarding Guide > Access and Permissions > VPN Access
  Note: Facilities question — not in the wiki corpus



Q: "tell me about quantum computing"
  Case: completely_off_topic
  Top final score: 0.413
  Top result: Data Privacy and Security Policy > Security Requirements > Data Encryption
  Note: Not related to any wiki content

Q: "deploy"
  Case: too_short
  Top final score: 0.818
  Top result: Production Deployment Guide > Deployment Windows
  Note: Very short query — ambiguous without context

Q: "how does the oncall thing work and what do I do if something brea"
  Case: too_long
  Top final score: 0.547
  Top result: Incident Response Playbook > On-Call Responsibilities > Primary On-Call
  Note: Very verbose query — embedding may dilute signal



In [21]:
# ── Export metrics ──

metrics = {
    "project": "Internal Wiki Assistant",
    "corpus": {
        "total_pages": len(WIKI_PAGES),
        "total_sections": len(all_sections),
        "departments": dict(Counter(
            p["metadata"]["department"] for p in WIKI_PAGES)),
        "avg_sections_per_page": len(all_sections) / len(WIKI_PAGES),
        "avg_section_length_chars": float(np.mean(
            [len(s.content) for s in all_sections])),
    },
    "retrieval_backend": wiki_index.backend,
    "evaluation": {
        "eval_queries": n_eval,
        "page_recall_at_1": page_hits / n_eval,
        "section_recall_at_3": section_hits / n_eval,
        "mrr": float(np.mean(mrr_values)),
        "keyword_page_recall_at_1": kw_page_hits / n_eval,
    },
    "config": {
        "top_k": TOP_K,
        "similarity_threshold": SIMILARITY_THRESHOLD,
        "embedding_model": EMBEDDING_MODEL if USE_ST else "TF-IDF",
        "recency_boost": RECENCY_BOOST,
        "tag_match_boost": TAG_MATCH_BOOST,
        "heading_depth_penalty": HEADING_DEPTH_PENALTY,
    },
}

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics, indent=2))

Metrics saved to E:\Github\Machine-Learning-Projects\GenAI\19_Internal_Wiki_Assistant\metrics.json
{
  "project": "Internal Wiki Assistant",
  "corpus": {
    "total_pages": 8,
    "total_sections": 89,
    "departments": {
      "Engineering": 5,
      "HR": 1,
      "Legal": 1,
      "Finance": 1
    },
    "avg_sections_per_page": 11.125,
    "avg_section_length_chars": 192.02247191011236
  },
  "retrieval_backend": "chroma",
  "evaluation": {
    "eval_queries": 10,
    "page_recall_at_1": 1.0,
    "section_recall_at_3": 1.0,
    "mrr": 1.0,
    "keyword_page_recall_at_1": 0.8
  },
  "config": {
    "top_k": 5,
    "similarity_threshold": 0.2,
    "embedding_model": "all-MiniLM-L6-v2",
    "recency_boost": 0.05,
    "tag_match_boost": 0.1,
    "heading_depth_penalty": 0.02
  }
}


## Limitations

1. **Tiny corpus** — 8 wiki pages is far too small for a real system. Real
   internal wikis have hundreds or thousands of pages. Index quality and
   retrieval accuracy may behave differently at scale.

2. **Flat metadata** — we use simple fields (department, tags, date). Real
   wikis have richer signals: page views, edit history, linked pages,
   permissions, and user feedback (helpful/not helpful).

3. **No cross-page linking** — wiki pages reference each other. A real
   system would follow links to surface related context across pages.

4. **No access control** — in production, search results must be filtered
   by the user's permissions. Not all employees should see all wiki content
   (e.g., salary bands, security audit reports).

5. **Stale detection is naive** — we boost recent documents, but a 2-year-old
   policy document may be more authoritative than a blog post from yesterday.
   Recency is a proxy, not ground truth.

6. **No answer synthesis** — we return raw wiki sections. A full assistant
   would use an LLM to synthesize a concise answer from the retrieved context.

7. **General-purpose embeddings** — our embedding model doesn't know Acme
   Corp's internal terminology. Fine-tuning on internal documents would
   significantly improve retrieval quality.

## Common Mistakes

1. **Indexing full pages instead of sections** — long pages dilute the
   embedding signal. Always chunk by heading structure for wiki content.

2. **Breaking sections mid-paragraph** — fixed-size chunking (e.g., 500
   chars) can split a rollback procedure in half. Heading-aware chunking
   preserves topical coherence.

3. **Ignoring heading hierarchy in search text** — a section's meaning depends
   on its parent headings. Including the breadcrumb path in the search text
   gives the embedding model crucial context.

4. **Not handling code blocks** — wiki pages often contain code snippets.
   Naive chunking may split a code block. Code-aware parsing should keep
   fenced code blocks intact within their section.

5. **Over-boosting metadata** — aggressive recency boosting can surface a
   trivially updated page (typo fix) over a genuinely relevant older one.
   Keep boost weights modest.

6. **No deduplication** — similar content in different pages can flood results.
   Consider deduplicating results from the same page.

## Mini Challenge

### Exercise 1: Link-Aware Retrieval
Add cross-references between wiki pages (e.g., "see Deployment Guide").
When a section references another page, include the linked content as
supplementary context in the response.

### Exercise 2: Freshness Indicators
Add visual freshness indicators to results:
- Green: updated within 6 months
- Yellow: 6-12 months old
- Red: >12 months old (may be stale)

### Exercise 3: Search Analytics
Track which queries return no results or low-confidence results. Use this
to identify gaps in the wiki that need new documentation.

### Exercise 4: Multi-Section Answers
For broad questions ("explain the incident process"), combine relevant
sections from the same page into a coherent multi-section answer.

### Exercise 5: Document Similarity Map
Build a similarity matrix between all wiki pages. Identify clusters and
suggest pages that should be linked or merged.

## Production Considerations

1. **Real wiki connectors** — integrate with Confluence API, Notion API,
   GitBook webhook, or monitor markdown files in a Git repo for changes.

2. **Incremental indexing** — re-index only changed pages, not the entire
   wiki. Use webhooks or polling for change detection.

3. **Access control** — filter results based on the user's permissions.
   Integrate with the wiki's permission system (Confluence spaces, Notion
   workspace membership).

4. **Slack/Teams integration** — deploy as a bot that answers questions
   directly in chat. Most wiki queries happen in conversation.

5. **Feedback loop** — let users rate results as helpful/not helpful.
   Use feedback to fine-tune ranking weights and identify content gaps.

6. **Version awareness** — track document versions. Show "this page was
   updated since your last visit" or "this answer is from a page last
   updated 18 months ago."

7. **Embedding model** — for production, consider fine-tuning an embedding
   model on your company's internal terminology and acronyms.

## How to Improve This Project

- **Add LLM synthesis** — use a language model to compose a concise answer
  from retrieved sections instead of returning raw text
- **Hybrid search** — combine BM25 keyword matching with semantic search
  for best-of-both-worlds retrieval
- **Query expansion** — expand acronyms and abbreviations automatically
  (e.g., "PR" → "pull request", "PTO" → "paid time off")
- **Table-aware parsing** — extract tables from markdown and index them
  separately for structured data queries
- **User personalization** — weight results based on the user's department
  and role (an engineer asking about "alerts" probably wants PagerDuty, not
  HR notifications)
- **Change monitoring** — alert subscribers when wiki pages they've accessed
  are updated

## Key Takeaways

1. **Section-level indexing beats page-level** — chunking at heading
   boundaries preserves topical coherence and points users to the exact
   answer, not just the page.

2. **Heading paths provide context** — including the breadcrumb
   (`"Deploy Guide > Rollback Procedure"`) in both the search text and
   the result display helps users understand and navigate results.

3. **Metadata improves ranking** — recency, tags, and department metadata
   provide signals beyond text similarity. A recently updated doc tagged
   with matching keywords should rank higher.

4. **Markdown structure is valuable** — unlike plain text, markdown has
   explicit heading hierarchy. Parsers should exploit this structure
   rather than treating the document as a flat string.

5. **Keep metadata boosts modest** — aggressive boosting can override
   genuine semantic relevance. The core signal should always be meaning
   similarity, with metadata as a refinement.

6. **Wiki search is a high-impact application** — employees spend
   significant time searching for internal information. A well-built
   wiki assistant directly reduces this friction.